# Day 3 — Training Pipelines & Subset Tests


---
## Step 1: Mount Drive & Install Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes wandb trl

---
## Step 2: Initialize Directories & Load Data

In [ ]:
import os

project_dir = "/content/Retail"
gdrive_dir = None
for candidate in ["/content/drive/MyDrive/Retail LLM", "/content/drive/MyDrive/Retail"]:
    if os.path.isdir(candidate):
        gdrive_dir = candidate
        break
if gdrive_dir is None:
    gdrive_dir = "/content/drive/MyDrive/Retail LLM"
print(f"[*] Using Drive folder: {gdrive_dir}")

for d in ["data/raw", "data/processed", "configs", "src", "models"]:
    os.makedirs(os.path.join(project_dir, d), exist_ok=True)

drive_processed = os.path.join(gdrive_dir, "data", "processed")
local_processed = os.path.join(project_dir, "data", "processed")
train_json = os.path.join(local_processed, "train.json")

if os.path.isdir(drive_processed):
    !cp -v "{drive_processed}/"*.json "{local_processed}/" 2>/dev/null || true

drive_raw = os.path.join(gdrive_dir, "data", "raw")
local_raw = os.path.join(project_dir, "data", "raw")
if os.path.isdir(drive_raw):
    !cp -v "{drive_raw}/"*.json "{local_raw}/" 2>/dev/null || true

if not os.path.exists(train_json):
    print("\n[!] Processed data NOT found. Running data_cleaning.py...")
    dc_code = "import os\nimport json\nimport random\n\ndef clean_text(text):\n    \"\"\"\n    Cleans and normalizes raw text input.\n    \"\"\"\n    if not isinstance(text, str):\n        return \"\"\n    \n    # Strip leading/trailing whitespaces and normalize internal spacing\n    text = \" \".join(text.split())\n    \n    # Standardize curly quotes and apostrophes to straight ones\n    text = text.replace(\"“\", \"\\\"\").replace(\"”\", \"\\\"\").replace(\"‘\", \"'\").replace(\"’\", \"'\")\n    \n    return text\n\ndef clean_and_split_data(raw_data_dir=\"data/raw\", processed_data_dir=\"data/processed\"):\n    \"\"\"\n    Cleans raw JSON data, converts to standard instruction-response pairs, \n    removes duplicates, shuffles deterministically, and splits 80/10/10.\n    \"\"\"\n    os.makedirs(processed_data_dir, exist_ok=True)\n    \n    # Files\n    retail_raw_path = os.path.join(raw_data_dir, \"retail_ecommerce_raw.json\")\n    mfg_raw_path = os.path.join(raw_data_dir, \"manufacturing_raw.json\")\n    \n    combined_pairs = []\n    \n    # Load and clean Retail data\n    print(\"[*] Loading Retail dataset...\")\n    if os.path.exists(retail_raw_path):\n        with open(retail_raw_path, \"r\", encoding=\"utf-8\") as f:\n            retail_data = json.load(f)\n            \n        for row in retail_data:\n            instruction = clean_text(row.get(\"instruction\", \"\"))\n            response = clean_text(row.get(\"response\", \"\"))\n            if instruction and response:\n                combined_pairs.append({\n                    \"instruction\": instruction,\n                    \"response\": response\n                })\n        print(f\"[+] Loaded {len(retail_data)} raw retail items.\")\n    else:\n        print(f\"[-] WARNING: Retail raw data file not found at {retail_raw_path}\")\n\n    # Load and clean Manufacturing data\n    print(\"[*] Loading Manufacturing dataset...\")\n    if os.path.exists(mfg_raw_path):\n        with open(mfg_raw_path, \"r\", encoding=\"utf-8\") as f:\n            mfg_data = json.load(f)\n            \n        for row in mfg_data:\n            instruction = clean_text(row.get(\"instruction\", \"\"))\n            response = clean_text(row.get(\"response\", \"\"))\n            if instruction and response:\n                combined_pairs.append({\n                    \"instruction\": instruction,\n                    \"response\": response\n                })\n        print(f\"[+] Loaded {len(mfg_data)} raw manufacturing items.\")\n    else:\n        print(f\"[-] WARNING: Manufacturing raw data file not found at {mfg_raw_path}\")\n\n    total_loaded = len(combined_pairs)\n    print(f\"[*] Total combined instruction-response pairs loaded: {total_loaded}\")\n    \n    # Deduplication\n    unique_pairs = []\n    seen_instructions = set()\n    for pair in combined_pairs:\n        # Deduplicate based on instruction content\n        if pair[\"instruction\"] not in seen_instructions:\n            seen_instructions.add(pair[\"instruction\"])\n            unique_pairs.append(pair)\n            \n    total_unique = len(unique_pairs)\n    print(f\"[+] Deduplication complete. Remaining unique records: {total_unique} (Removed {total_loaded - total_unique} duplicates).\")\n    \n    # Deterministic Shuffle for reproducibility\n    print(\"[*] Shuffling dataset deterministically...\")\n    random.seed(42)\n    random.shuffle(unique_pairs)\n    \n    # Split 80 / 10 / 10\n    total = len(unique_pairs)\n    train_end = int(total * 0.8)\n    val_end = train_end + int(total * 0.1)\n    \n    train_split = unique_pairs[:train_end]\n    val_split = unique_pairs[train_end:val_end]\n    test_split = unique_pairs[val_end:]\n    \n    print(f\"\\n[+] Split splits count:\")\n    print(f\"    - Train Split (80%): {len(train_split)} items\")\n    print(f\"    - Val Split (10%): {len(val_split)} items\")\n    print(f\"    - Test Split (10%): {len(test_split)} items\")\n    \n    # Save splits\n    splits = {\n        \"train.json\": train_split,\n        \"val.json\": val_split,\n        \"test.json\": test_split\n    }\n    \n    for filename, split_data in splits.items():\n        output_path = os.path.join(processed_data_dir, filename)\n        with open(output_path, \"w\", encoding=\"utf-8\") as f:\n            json.dump(split_data, f, ensure_ascii=False, indent=2)\n        print(f\"[+] Saved split file: {output_path}\")\n\nif __name__ == \"__main__\":\n    # Clean and split locally if run directly\n    clean_and_split_data()\n"
    with open("/content/Retail/src/data_cleaning.py", "w", encoding="utf-8") as f:
        f.write(dc_code)
    %cd /content/Retail
    !python src/data_cleaning.py
    %cd /content

for fname in ["train.json", "val.json", "test.json"]:
    fpath = os.path.join(local_processed, fname)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024*1024)
        print(f"[+] {fname}: {size_mb:.2f} MB")
    else:
        print(f"[X] MISSING: {fname}")
print("\n[+] Data setup complete.")

---
## Step 3: Write Configuration & Script Files

In [ ]:
qwen_cfg = "{\n  \"model_type\": \"qwen\",\n  \"base_model_name_or_path\": \"Qwen/Qwen2.5-7B-Instruct\",\n  \"peft_config\": {\n    \"r\": 16,\n    \"lora_alpha\": 32,\n    \"lora_dropout\": 0.05,\n    \"bias\": \"none\",\n    \"task_type\": \"CAUSAL_LM\",\n    \"target_modules\": [\n      \"q_proj\",\n      \"k_proj\",\n      \"v_proj\",\n      \"o_proj\",\n      \"gate_proj\",\n      \"up_proj\",\n      \"down_proj\"\n    ]\n  },\n  \"quantization_config\": {\n    \"load_in_4bit\": true,\n    \"bnb_4bit_quant_type\": \"nf4\",\n    \"bnb_4bit_use_double_quant\": true,\n    \"bnb_4bit_compute_dtype\": \"float16\"\n  }\n}\n";
with open("/content/Retail/configs/qwen_lora_config.json", "w", encoding="utf-8") as f:
    f.write(qwen_cfg)
print("[+] configs/qwen_lora_config.json")

llama_cfg = "{\n  \"model_type\": \"llama\",\n  \"base_model_name_or_path\": \"meta-llama/Meta-Llama-3-8B-Instruct\",\n  \"peft_config\": {\n    \"r\": 16,\n    \"lora_alpha\": 32,\n    \"lora_dropout\": 0.05,\n    \"bias\": \"none\",\n    \"task_type\": \"CAUSAL_LM\",\n    \"target_modules\": [\n      \"q_proj\",\n      \"k_proj\",\n      \"v_proj\",\n      \"o_proj\",\n      \"gate_proj\",\n      \"up_proj\",\n      \"down_proj\"\n    ]\n  },\n  \"quantization_config\": {\n    \"load_in_4bit\": true,\n    \"bnb_4bit_quant_type\": \"nf4\",\n    \"bnb_4bit_use_double_quant\": true,\n    \"bnb_4bit_compute_dtype\": \"float16\"\n  }\n}\n";
with open("/content/Retail/configs/llama_lora_config.json", "w", encoding="utf-8") as f:
    f.write(llama_cfg)
print("[+] configs/llama_lora_config.json")

In [ ]:
train_code = "import os\nimport argparse\nimport json\nimport torch\nfrom datasets import load_dataset\nfrom transformers import (\n    AutoModelForCausalLM,\n    AutoTokenizer,\n    BitsAndBytesConfig\n)\nfrom peft import (\n    LoraConfig,\n    get_peft_model,\n    prepare_model_for_kbit_training\n)\nfrom trl import SFTTrainer, SFTConfig\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description=\"QLoRA Fine-Tuning Pipeline for Retail & Manufacturing LLMs\")\n    parser.add_argument(\n        \"--config\",\n        type=str,\n        required=True,\n        help=\"Path to the model configuration JSON file (e.g. configs/qwen_lora_config.json)\"\n    )\n    parser.add_argument(\n        \"--train_file\",\n        type=str,\n        default=\"data/processed/train.json\",\n        help=\"Path to the training data file\"\n    )\n    parser.add_argument(\n        \"--val_file\",\n        type=str,\n        default=\"data/processed/val.json\",\n        help=\"Path to the validation data file\"\n    )\n    parser.add_argument(\n        \"--output_dir\",\n        type=str,\n        default=None,\n        help=\"Directory to save the fine-tuned model checkpoints (defaults to models/{model_type}_v1)\"\n    )\n    parser.add_argument(\n        \"--test_subset\",\n        action=\"store_true\",\n        help=\"If set, runs a quick training check on a tiny dataset slice (50 items) for 5 steps.\"\n    )\n    return parser.parse_args()\n\ndef format_example(example):\n    \"\"\"\n    Converts a single dataset row into an instruction-tuning prompt string.\n    \"\"\"\n    instruction = example['instruction']\n    response = example['response']\n    example['text'] = (\n        f\"Below is an instruction that describes a task. Write a response that appropriately completes the request.\\n\\n\"\n        f\"### Instruction:\\n{instruction}\\n\\n\"\n        f\"### Response:\\n{response}\"\n    )\n    return example\n\ndef main():\n    args = parse_args()\n    \n    # 1. Load Model Settings Configuration\n    print(f\"[*] Loading configuration from: {args.config}\")\n    if not os.path.exists(args.config):\n        raise FileNotFoundError(f\"[-] Config file not found at {args.config}\")\n        \n    with open(args.config, \"r\", encoding=\"utf-8\") as f:\n        config = json.load(f)\n        \n    model_type = config.get(\"model_type\", \"model\")\n    model_name = config.get(\"base_model_name_or_path\")\n    peft_settings = config.get(\"peft_config\", {})\n    quant_settings = config.get(\"quantization_config\", {})\n    \n    # Auto-assign output directory if not provided\n    if args.output_dir is None:\n        args.output_dir = f\"models/{model_type}_v1\"\n    print(f\"[+] Output Directory: {args.output_dir}\")\n    \n    # 2. Setup Quantization Configuration\n    compute_dtype_str = quant_settings.get(\"bnb_4bit_compute_dtype\", \"bfloat16\")\n    compute_dtype = torch.bfloat16 if compute_dtype_str == \"bfloat16\" else torch.float16\n    \n    if not torch.cuda.is_available():\n        raise RuntimeError(\"[-] CUDA is not available! QLoRA training requires an active GPU runtime.\")\n        \n    bnb_config = BitsAndBytesConfig(\n        load_in_4bit=quant_settings.get(\"load_in_4bit\", True),\n        bnb_4bit_quant_type=quant_settings.get(\"bnb_4bit_quant_type\", \"nf4\"),\n        bnb_4bit_use_double_quant=quant_settings.get(\"bnb_4bit_use_double_quant\", True),\n        bnb_4bit_compute_dtype=compute_dtype\n    )\n    \n    # 3. Load Tokenizer & Model\n    print(f\"[*] Loading tokenizer for {model_name}...\")\n    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)\n    tokenizer.padding_side = \"right\" # SFTTrainer requires padding side right\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n        \n    print(f\"[*] Loading base model {model_name} in 4-bit quantization (this will take a few minutes)...\")\n    model = AutoModelForCausalLM.from_pretrained(\n        model_name,\n        quantization_config=bnb_config,\n        device_map=\"auto\",\n        trust_remote_code=True,\n        torch_dtype=torch.float16\n    )\n    print(\"[+] Base model loaded in 4-bit.\")\n    \n    # 1. Force all parameters and buffers in the base model to float16 to prevent bfloat16 propagation\n    for name, param in model.named_parameters():\n        if param.dtype == torch.bfloat16:\n            param.data = param.data.to(torch.float16)\n    for name, buf in model.named_buffers():\n        if buf.dtype == torch.bfloat16:\n            buf.data = buf.data.to(torch.float16)\n            \n    # 2. Set model config torch_dtype to float32 so PEFT initializes adapters in float32\n    model.config.torch_dtype = torch.float32\n            \n    # 4. Prepare Model for PEFT/LoRA Training\n    model = prepare_model_for_kbit_training(model)\n    \n    # 5. Configure LoRA\n    print(\"[*] Configuring LoRA Adapter...\")\n    lora_config = LoraConfig(\n        r=peft_settings.get(\"r\", 16),\n        lora_alpha=peft_settings.get(\"lora_alpha\", 32),\n        target_modules=peft_settings.get(\"target_modules\", []),\n        lora_dropout=peft_settings.get(\"lora_dropout\", 0.05),\n        bias=peft_settings.get(\"bias\", \"none\"),\n        task_type=\"CAUSAL_LM\"\n    )\n    # NOTE: We do NOT call get_peft_model() here — SFTTrainer applies it via peft_config\n    print(\"[+] LoRA config ready.\")\n    \n    # 6. Load Dataset\n    print(f\"[*] Loading dataset files: {args.train_file} & {args.val_file}...\")\n    dataset = load_dataset(\n        \"json\",\n        data_files={\n            \"train\": args.train_file,\n            \"validation\": args.val_file\n        }\n    )\n    \n    train_dataset = dataset[\"train\"].map(format_example)\n    val_dataset = dataset[\"validation\"].map(format_example)\n    print(f\"[+] Formatted datasets with 'text' column.\")\n    \n    # 7. Configure Training Arguments\n    if args.test_subset:\n        print(\"\\n==============================================\")\n        print(\"[!] TEST MODE ENABLED: Slicing datasets and steps\")\n        print(\"==============================================\")\n        train_dataset = train_dataset.select(range(min(len(train_dataset), 50)))\n        val_dataset = val_dataset.select(range(min(len(val_dataset), 10)))\n        print(f\"[+] Sliced datasets: Train = {len(train_dataset)} | Val = {len(val_dataset)}\")\n        \n        training_args = SFTConfig(\n            output_dir=args.output_dir,\n            dataset_text_field=\"text\",\n            max_length=512,\n            per_device_train_batch_size=2,\n            per_device_eval_batch_size=2,\n            gradient_accumulation_steps=1,\n            max_steps=5, # Run only 5 steps to verify loop\n            learning_rate=2e-4,\n            logging_steps=1,\n            eval_strategy=\"steps\",\n            eval_steps=1,\n            save_strategy=\"no\",\n            fp16=True,\n            report_to=\"none\", # Disable W&B logging for quick tests\n            remove_unused_columns=False,\n            disable_tqdm=False\n        )\n    else:\n        print(f\"[+] Datasets loaded: Train = {len(train_dataset)} | Val = {len(val_dataset)}\")\n        training_args = SFTConfig(\n            output_dir=args.output_dir,\n            dataset_text_field=\"text\",\n            max_length=512,\n            num_train_epochs=3,\n            per_device_train_batch_size=4,\n            per_device_eval_batch_size=4,\n            gradient_accumulation_steps=4, # Effective batch size = 16\n            learning_rate=2e-4,\n            logging_steps=10,\n            eval_strategy=\"steps\",\n            eval_steps=50,\n            save_strategy=\"steps\",\n            save_steps=100,\n            save_total_limit=1,\n            fp16=True,\n            report_to=\"wandb\" if os.environ.get(\"WANDB_DISABLED\", \"\").lower() != \"true\" else \"none\",\n            warmup_ratio=0.03,\n            lr_scheduler_type=\"cosine\",\n            remove_unused_columns=False\n        )\n\n    # 8. Initialize SFTTrainer\n    print(\"[*] Initializing SFTTrainer...\")\n    trainer = SFTTrainer(\n        model=model,\n        train_dataset=train_dataset,\n        eval_dataset=val_dataset,\n        peft_config=lora_config,\n        processing_class=tokenizer,\n        args=training_args\n    )\n    \n    # Force cast any remaining bfloat16 parameters or buffers inside the trainer model to float32\n    # to prevent bfloat16 gradient scaling crash on T4 GPU.\n    for name, param in trainer.model.named_parameters():\n        if param.dtype == torch.bfloat16:\n            param.data = param.data.to(torch.float32)\n    for name, buf in trainer.model.named_buffers():\n        if buf.dtype == torch.bfloat16:\n            buf.data = buf.data.to(torch.float32)\n            \n    # 9. Launch Training\n    print(\"[*] Starting training...\")\n    trainer.train()\n    print(\"[+] Training completed successfully!\")\n    \n    # Save final adapter weights\n    print(f\"[*] Saving adapter weights to: {args.output_dir}\")\n    trainer.model.save_pretrained(args.output_dir)\n    tokenizer.save_pretrained(args.output_dir)\n    print(\"[+] Saving complete.\")\n\nif __name__ == \"__main__\":\n    main()\n";
with open("/content/Retail/src/train.py", "w", encoding="utf-8") as f:
    f.write(train_code)
print("[+] src/train.py")

---
## Step 4: W&B (Disabled for Testing)

In [ ]:
import os
os.environ['WANDB_DISABLED'] = 'true'
print("[+] W&B disabled for subset tests.")

---
## Step 5: Test Qwen Training (Subset Mode)

In [ ]:
!python /content/Retail/src/train.py \
    --config /content/Retail/configs/qwen_lora_config.json \
    --train_file /content/Retail/data/processed/train.json \
    --val_file /content/Retail/data/processed/val.json \
    --test_subset

---
## Step 6: Test Llama Training (Subset Mode)

In [ ]:
from google.colab import userdata
import os

if not os.environ.get('HF_TOKEN'):
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
        print("[+] Loaded HF_TOKEN.")
    except Exception:
        print("[-] HF_TOKEN not found.")

!python /content/Retail/src/train.py \
    --config /content/Retail/configs/llama_lora_config.json \
    --train_file /content/Retail/data/processed/train.json \
    --val_file /content/Retail/data/processed/val.json \
    --test_subset

---
## Step 7: Backup to Google Drive

In [ ]:
import os
gdrive_target = "/content/drive/MyDrive/Retail LLM"
for candidate in ["/content/drive/MyDrive/Retail LLM", "/content/drive/MyDrive/Retail"]:
    if os.path.isdir(candidate):
        gdrive_target = candidate
        break
os.makedirs(gdrive_target, exist_ok=True)
!rsync -av --progress /content/Retail/ "{gdrive_target}/"
print("[+] Backup complete.")